# Loading + Preprocessing functions and alignment functions

In [3]:
BASE_PATH                   = '/Users/francescocenciarelli/Desktop/daly_2019/sub-02'
MIDI_FEATURES_PATH          = '/Users/francescocenciarelli/Desktop/MIND_Hack/MIND_hackathon/midi_player/midi_features.npz'

In [ ]:
import mne
import nibabel as nib
import pandas as pd
import json
import numpy as np
import warnings
from scipy.signal import find_peaks
import matplotlib.pyplot as plt


def create_activation_maps(data):
    """
    Generates binary activation maps for each timepoint in a 4D NIfTI dataset.
    
    INPUT:
        data: 4D numpy array (Y x X x Z x T), expected to be float type.
        
    OUTPUT:
        activation_maps: 4D binary numpy array (Y x X x Z x T), where 1 indicates activation.
    """
    # Get the shape of the data
    Y, X, Z, T = data.shape
    
    # Preallocate output array
    activation_maps = np.zeros((Y, X, Z, T), dtype=bool)  # boolean array
    
    # Loop over timepoints
    for t in range(T):
        # Extract the 3D volume at time t
        volume = data[:, :, :, t]
        
        # Flatten the volume to a vector
        volume_vector = volume.flatten()
        
        # Find the 80th percentile value (top 20% of voxels)
        threshold = np.percentile(volume_vector, 80)  # Top 20% voxels
        
        # Create binary activation map
        activation_map = volume > threshold
        
        # Store the activation map
        activation_maps[:, :, :, t] = activation_map
    
    return activation_maps

def align_eeg_and_fmri_sub02(run, base=BASE_PATH):
    """
    Align sub-02's EEG to fMRI and load both datasets, printing shapes
    before/after each major step.

    Also loads MIDI features and creates a time-aligned feature array.

    Returns:
      - data_eeg:   ndarray, shape (n_channels, n_samples)
      - sfreq:      float, EEG sampling rate
      - fmri_data:  ndarray, shape (X, Y, Z, n_vols)
      - TR:         float, repetition time in seconds
      - music_song: ndarray, shape (n_samples, 3) - aligned song features
    """
    print(f"\n=== Aligning run: {run} ===")
    # Paths
    eeg_f = f"{base}/eeg/sub-02_task-{run}_eeg.edf"
    evt   = f"{base}/eeg/sub-02_task-{run}_events.tsv"
    bold  = f"{base}/func/sub-02_task-{run}_bold.nii.gz"
    jsn   = f"{base}/func/sub-02_task-{run}_bold.json"
    midi_features_path = MIDI_FEATURES_PATH  # Define path for MIDI features
    
    # --- Load MIDI Features --- 
    try:
        midi_data = np.load(midi_features_path)
        midi_features = midi_data['features']  # Shape (216, 3, 40000)
        midi_ids = midi_data['ids']            # Shape (216,)
        print(f"Loaded MIDI features from {midi_features_path}")
        # Create a mapping from song ID to its index in the features array
        id_to_feature_index = {
            int(id_val): idx for idx, id_val in enumerate(midi_ids)
        }
        print(f"Created mapping for {len(id_to_feature_index)} unique MIDI IDs.")
    except FileNotFoundError:
        print(f"Error: MIDI features file not found at {midi_features_path}")
        return None, None, None, None, None # Or raise an error
    except KeyError as e:
        print(f"Error: Missing key '{e}' in {midi_features_path}")
        return None, None, None, None, None
    # --- End MIDI Feature Loading ---

    # 1) Load raw EEG
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        raw = mne.io.read_raw_edf(eeg_f, preload=True, verbose=False)
    print(f"Raw EEG: {raw.info['nchan']} channels × {raw.n_times} samples")
    
    # 1b) extract the music‐label channel by known index (0-based 40)
    m_idx = 38
    music_full = raw.get_data()[m_idx, :].copy() # Use copy to avoid modifying raw data
    print(f"Extracted music channel index {m_idx}: '{raw.ch_names[m_idx]}'")
    print(f"Music channel shape: {music_full.shape}")

    # === USER REQUEST: Multiply non-zero values ===
    scaling_factor = 20 * 1000000
    # Operate only on non-zero elements before scaling
    non_zero_mask = music_full != 0
    music_full[non_zero_mask] *= scaling_factor
    music_full = np.round(music_full)
    print(f"Amount of non-zero samples after scaling/rounding: "
          f"{np.sum(music_full != 0)}")
    # ==============================================

    # Find peaks on the scaled/rounded signal (with plateaus)
    peaks, _ = find_peaks(music_full, distance=600)
    # Get the actual values (song IDs) at these peak locations
    peak_song_ids = music_full[peaks].astype(int) # Ensure IDs are integers

    # Create a sparse array: zeros everywhere except at peaks
    music_sparse = np.zeros_like(music_full)
    music_sparse[peaks] = peak_song_ids  # Place song IDs at peak indices

    # Replace the plateau version with the sparse version for clarity downstream
    music_full = music_sparse # Now music_full is the sparse array

    print(f"Non-zero values (Song IDs) after sparsification: {peak_song_ids}")
    # This count should now match the number of peaks
    print(f"Amount of non-zero samples after sparsification (num peaks): "
          f"{len(peaks)}")

    # --- Create Aligned Song Feature Array --- 
    print("Creating aligned song feature array...")
    # Initialize with zeros, shape (n_samples, 3), matching feature dtype
    music_song = np.zeros((len(music_full), 3), dtype=midi_features.dtype)
    feature_len = 40000  # Expected length of each song feature timeseries

    for peak_idx, song_id in zip(peaks, peak_song_ids):
        feature_idx = id_to_feature_index.get(song_id)
        
        if feature_idx is not None:
            # Extract the (3, 40000) feature slice for this song ID
            feature_slice = midi_features[feature_idx, :, :]
            
            # Determine insertion range in music_song
            start_idx = peak_idx
            end_idx = start_idx + feature_len
            
            # Handle boundary condition: if feature goes past end of signal
            if end_idx > music_song.shape[0]:
                print(f"  Warning: Feature for ID {song_id} at index {peak_idx} extends beyond signal length.")
                # Truncate the slice length
                actual_end_idx = music_song.shape[0]
                slice_len = actual_end_idx - start_idx
                # Insert the truncated, transposed features
                music_song[start_idx:actual_end_idx, :] = feature_slice[:, :slice_len].T
            else:
                # Insert the full, transposed features (40000, 3)
                music_song[start_idx:end_idx, :] = feature_slice.T
        else:
            print(f"  Warning: Song ID {song_id} found in EEG data at index {peak_idx} has no matching feature in MIDI data.")
            
    print(f"Created music_song array, shape: {music_song.shape}")
    # --- End Feature Array Creation --- 

    # 2) pick only the first 32 EEG channels
    picks = list(range(32))
    raw.pick(picks)
    print(f" After picking 32 EEG channels: {raw.info['nchan']} channels × "
          f"{raw.n_times} samples")
    sfreq = raw.info['sfreq']
    
    # filtering
    raw.notch_filter(50.0, picks='eeg', filter_length='auto',
                     phase='zero', fir_design='firwin')
    raw.filter(l_freq=0.5, h_freq=50.0, picks='eeg',
               filter_length='auto', phase='zero', fir_design='firwin')
    print(f" After filtering: {raw.info['nchan']} channels × "
          f"{raw.n_times} samples")
    
    # 3) Find first TTL
    ev  = pd.read_csv(evt, sep='\t')
    ttl = ev[ev['trial_type'] == 10]
    first_onset = ttl.iloc[0]['onset']  # in seconds
    print(" First TTL onset (s):", first_onset)
    
    # 4) Convert to sample index
    first_ttl = int(round(first_onset * sfreq))
    print(" TTL sample index:", first_ttl)
    
    # 5) Trim EEG before TTL
    data_eeg = raw.get_data()[:, first_ttl:]
    music_song = music_song[first_ttl:, :] # Trim music_song to match EEG start
    print(f" After trim EEG: {data_eeg.shape[0]} channels × {data_eeg.shape[1]} samples")
    print(f" After trim music_song: {music_song.shape}")
    
    # 6) Load fMRI 4D data and metadata
    img      = nib.load(bold)
    fmri_data = img.get_fdata()  
    print(f" Loaded fMRI data shape: {fmri_data.shape}")  # (X×Y×Z×n_vols)
    activation_maps = create_activation_maps(fmri_data)
    fmri_data = fmri_data * activation_maps
    print(f" masked fMRI data shape: {fmri_data.shape}")  # (X×Y×Z×n_vols)
    n_vols   = fmri_data.shape[3]
    with open(jsn) as f:
        md = json.load(f)
    TR       = float(md['RepetitionTime'])
    print(" fMRI vols:", n_vols, "   TR(s):", TR)
    
    # 7) Compute needed EEG samples
    needed = int(round(n_vols * TR * sfreq))
    print(" Needed EEG samples:", needed)
    
    # 8) Pad or crop EEG AND music_song to match fMRI duration
    if data_eeg.shape[1] < needed:
        pad = needed - data_eeg.shape[1]
        data_eeg = np.pad(data_eeg, ((0,0),(0,pad)), mode='constant')
        music_song = np.pad(music_song, ((0, pad), (0, 0)), mode='constant')
        print(f" After PAD EEG: {data_eeg.shape[1]} samples (+{pad} pad)")
        print(f" After PAD music_song: {music_song.shape[0]} samples")
    elif data_eeg.shape[1] > needed:
        data_eeg = data_eeg[:, :needed]
        music_song = music_song[:needed, :]
        print(f" After CROP EEG: {data_eeg.shape[1]} samples (cropped)")
        print(f" After CROP music_song: {music_song.shape[0]} samples")
    else:
        print(" EEG and music_song lengths already match needed duration.")

    print(f"✔ Final aligned EEG shape: {data_eeg.shape[0]} channels × "
          f"{data_eeg.shape[1]} samples")
    print(f"✔ Final aligned music_song shape: {music_song.shape}")
    print(f"✔ Final fMRI    shape: {fmri_data.shape}")
    
    return data_eeg, sfreq, fmri_data, TR, music_song


def segment_trials_sub02(data_eeg, sfreq, fmri_data, TR,
                         music_song, # Add music_song as input
                         events_tsv,
                         trial_duration_s=40.0,
                         ttl_code=768):
    """
    Segment the already-aligned EEG (n_chan × n_samp), 
    fMRI (X×Y×Z×n_vols), and music_song (n_samp × 3) 
    into trial-long chunks of exactly `trial_duration_s` seconds,
    using event code `ttl_code` in `events_tsv`.

    Returns:
      - trials_eeg:  ndarray, shape (n_trials, n_chan, samples_per_trial)
      - trials_fmri: ndarray, shape (n_trials, X, Y, Z, vols_per_trial)
      - trials_song: ndarray, shape (n_trials, samples_per_trial, 3)
    """
    if sfreq is None:
        raise ValueError("sfreq (sampling frequency) must not be None")
    if TR is None:
        raise ValueError("TR (repetition time) must not be None")
    midi_features_path = MIDI_FEATURES_PATH
    # load events
    ev = pd.read_csv(events_tsv, sep='\t')
    # find onsets of "Trial start"
    onsets = ev.query("trial_type == @ttl_code")['onset'].values  # in seconds
    print(f"Found {len(onsets)} trial starts at:\n {onsets}")

    samples_per_trial = int(round(trial_duration_s * sfreq))
    vols_per_trial   = int(round(trial_duration_s / TR))

    print(f"Each trial → {samples_per_trial} EEG samples | "
          f"{vols_per_trial} fMRI vols")

    eeg_trials  = []
    fmri_trials = []
    song_trials = [] # List to hold song feature trials

    for t0 in onsets:
        # EEG slice
        samp0 = int(round(t0 * sfreq))
        e_seg = data_eeg[:, samp0 : samp0 + samples_per_trial]
        if e_seg.shape[1] != samples_per_trial:
            print(f"  • skipping EEG at {t0}s → only {e_seg.shape[1]} samples")
            continue

        # fMRI slice
        vol0 = int(round(t0 / TR))
        f_seg = fmri_data[..., vol0 : vol0 + vols_per_trial]
        if f_seg.shape[3] != vols_per_trial:
            print(f"  • skipping fMRI at {t0}s → only {f_seg.shape[3]} vols")
            continue

        # music_song slice
        # NOTE: We assume music_song is aligned with EEG samples
        s_seg = music_song[samp0: samp0 + samples_per_trial, :]
        if s_seg.shape[0] != samples_per_trial:
            print(f"  • skipping music_song at {t0}s → only {s_seg.shape[0]} samples")
            continue

        eeg_trials.append(e_seg)
        fmri_trials.append(f_seg)
        song_trials.append(s_seg) # Append song segment

    trials_eeg = np.stack(eeg_trials, axis=0)
    trials_fmri = np.stack(fmri_trials, axis=0)
    trials_song = np.stack(song_trials, axis=0) # Stack song trials

    # --- Check for zero-value periods within trials_song --- 
    found_zero_periods = False
    print("\nChecking for zero-value periods within trials_song...")
    for trial_idx in range(trials_song.shape[0]):
        trial_data = trials_song[trial_idx]  # Shape (samples_per_trial, 3)
        # Find samples where all 3 features are zero
        is_zero_sample = np.all(trial_data == 0, axis=1)

        if not np.any(is_zero_sample):
            # Skip trial if it contains no zero samples at all
            continue

        # Find the start and end indices of contiguous blocks of zeros
        diff = np.diff(is_zero_sample.astype(int))
        starts = np.where(diff == 1)[0] + 1
        ends = np.where(diff == -1)[0]  # End is inclusive index of last zero

        # Handle edge cases: zero block at the start or end
        if is_zero_sample[0]:
            starts = np.insert(starts, 0, 0)
        if is_zero_sample[-1]:
            # If the last sample is zero, the block extends to the end
            # Need to find the corresponding start for the last block
            if len(ends) < len(starts):
                 ends = np.append(ends, len(is_zero_sample) - 1)

        # Ensure we have matching starts and ends
        min_len = min(len(starts), len(ends))
        starts = starts[:min_len]
        ends = ends[:min_len]

        if len(starts) > 0: # If any zero blocks were identified
             if not found_zero_periods:
                 print("  Warning: Found periods of zero values within trials:")
                 found_zero_periods = True
             print(f"    Trial {trial_idx}:")
             for start, end in zip(starts, ends):
                 # end index from diff is inclusive, add 1 for length
                 length = end - start + 1
                 if length > 0: # Print only valid blocks
                     print(f"      - Zero block from sample {start} to {end} (length {length})")

    if not found_zero_periods:
         print("  No significant zero-value periods found within any trial.")
    # --- End check --- 

    print("\n✔ Successfully segmented:")
    print(f"  • trials_eeg shape : {trials_eeg.shape}")
    print("    (n_trials, n_chan, samples)")
    print(f"  • trials_fmri shape: {trials_fmri.shape}")
    print("    (n_trials, X, Y, Z, vols)")
    print(f"  • trials_song shape: {trials_song.shape}") # Print shape
    print("    (n_trials, samples, 3)")

    return trials_eeg, trials_fmri, trials_song # Return song trials

# Run it and watch the printouts
runs = ['genMusic01','genMusic02','genMusic03','classicalMusic','washout']
for run in runs:
   eeg_aligned, sf, fmri_4d, TR, music_song = align_eeg_and_fmri_sub02(run)

data_eeg, sfreq, fmri_4d, TR, music_song = align_eeg_and_fmri_sub02('genMusic01')



=== Aligning run: genMusic01 ===
Loaded MIDI features from /Users/francescocenciarelli/Desktop/MIND_Hack/MIND_hackathon/midi_player/midi_features.npz
Created mapping for 216 unique MIDI IDs.
Raw EEG: 47 channels × 600000 samples
Extracted music channel index 38: 'music'
Music channel shape: (600000,)
Amount of non-zero samples after scaling/rounding: 11000
Non-zero values (Song IDs) after sparsification: [752 822 753 753 452 753 543 281 751 283 751]
Amount of non-zero samples after sparsification (num peaks): 11
Creating aligned song feature array...
Created music_song array, shape: (600000, 3)
 After picking 32 EEG channels: 32 channels × 600000 samples
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edg

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s


 After filtering: 32 channels × 600000 samples
 First TTL onset (s): 35.683
 TTL sample index: 35683
 After trim EEG: 32 channels × 564317 samples
 After trim music_song: (564317, 3)
 Loaded fMRI data shape: (64, 64, 37, 274)
 masked fMRI data shape: (64, 64, 37, 274)
 fMRI vols: 274    TR(s): 2.0
 Needed EEG samples: 548000
 After CROP EEG: 548000 samples (cropped)
 After CROP music_song: 548000 samples
✔ Final aligned EEG shape: 32 channels × 548000 samples
✔ Final aligned music_song shape: (548000, 3)
✔ Final fMRI    shape: (64, 64, 37, 274)

=== Aligning run: genMusic02 ===
Loaded MIDI features from /Users/francescocenciarelli/Desktop/MIND_Hack/MIND_hackathon/midi_player/midi_features.npz
Created mapping for 216 unique MIDI IDs.
Raw EEG: 47 channels × 594000 samples
Extracted music channel index 38: 'music'
Music channel shape: (594000,)
Amount of non-zero samples after scaling/rounding: 8000
Non-zero values (Song IDs) after sparsification: [ 283  282  281  821 1646  752  823  823]

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s


 After filtering: 32 channels × 594000 samples
 First TTL onset (s): 23.118
 TTL sample index: 23118
 After trim EEG: 32 channels × 570882 samples
 After trim music_song: (570882, 3)
 Loaded fMRI data shape: (64, 64, 37, 276)
 masked fMRI data shape: (64, 64, 37, 276)
 fMRI vols: 276    TR(s): 2.0
 Needed EEG samples: 552000
 After CROP EEG: 552000 samples (cropped)
 After CROP music_song: 552000 samples
✔ Final aligned EEG shape: 32 channels × 552000 samples
✔ Final aligned music_song shape: (552000, 3)
✔ Final fMRI    shape: (64, 64, 37, 276)

=== Aligning run: genMusic03 ===
Loaded MIDI features from /Users/francescocenciarelli/Desktop/MIND_Hack/MIND_hackathon/midi_player/midi_features.npz
Created mapping for 216 unique MIDI IDs.
Raw EEG: 47 channels × 593000 samples
Extracted music channel index 38: 'music'
Music channel shape: (593000,)
Amount of non-zero samples after scaling/rounding: 6000
Non-zero values (Song IDs) after sparsification: [823 822 753 283 436]
Amount of non-zero 

...

 After filtering: 32 channels × 1898000 samples
 First TTL onset (s): 15.68
 TTL sample index: 15680
 After trim EEG: 32 channels × 1882320 samples
 After trim music_song: (1882320, 3)
 Loaded fMRI data shape: (64, 64, 37, 934)
 masked fMRI data shape: (64, 64, 37, 934)
 fMRI vols: 934    TR(s): 2.0
 Needed EEG samples: 1868000
 After CROP EEG: 1868000 samples (cropped)
 After CROP music_song: 1868000 samples
✔ Final aligned EEG shape: 32 channels × 1868000 samples
✔ Final aligned music_song shape: (1868000, 3)
✔ Final fMRI    shape: (64, 64, 37, 934)

=== Aligning run: washout ===
Loaded MIDI features from /Users/francescocenciarelli/Desktop/MIND_Hack/MIND_hackathon/midi_player/midi_features.npz
Created mapping for 216 unique MIDI IDs.
Raw EEG: 47 channels × 112000 samples
Extracted music channel index 38: 'music'
Music channel shape: (112000,)
Amount of non-zero samples after scaling/rounding: 0
Non-zero values (Song IDs) after sparsification: []
Amount of non-zero samples after spar

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


 First TTL onset (s): 23.114
 TTL sample index: 23114
 After trim EEG: 32 channels × 88886 samples
 After trim music_song: (88886, 3)
 Loaded fMRI data shape: (64, 64, 37, 36)
 masked fMRI data shape: (64, 64, 37, 36)
 fMRI vols: 36    TR(s): 2.0
 Needed EEG samples: 72000
 After CROP EEG: 72000 samples (cropped)
 After CROP music_song: 72000 samples
✔ Final aligned EEG shape: 32 channels × 72000 samples
✔ Final aligned music_song shape: (72000, 3)
✔ Final fMRI    shape: (64, 64, 37, 36)

=== Aligning run: genMusic01 ===
Loaded MIDI features from /Users/francescocenciarelli/Desktop/MIND_Hack/MIND_hackathon/midi_player/midi_features.npz
Created mapping for 216 unique MIDI IDs.
Raw EEG: 47 channels × 600000 samples
Extracted music channel index 38: 'music'
Music channel shape: (600000,)
Amount of non-zero samples after scaling/rounding: 11000
Non-zero values (Song IDs) after sparsification: [752 822 753 753 452 753 543 281 751 283 751]
Amount of non-zero samples after sparsification (num

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s


 After filtering: 32 channels × 600000 samples
 First TTL onset (s): 35.683
 TTL sample index: 35683
 After trim EEG: 32 channels × 564317 samples
 After trim music_song: (564317, 3)
 Loaded fMRI data shape: (64, 64, 37, 274)
 masked fMRI data shape: (64, 64, 37, 274)
 fMRI vols: 274    TR(s): 2.0
 Needed EEG samples: 548000
 After CROP EEG: 548000 samples (cropped)
 After CROP music_song: 548000 samples
✔ Final aligned EEG shape: 32 channels × 548000 samples
✔ Final aligned music_song shape: (548000, 3)
✔ Final fMRI    shape: (64, 64, 37, 274)


# Model functions and training loop

In [ ]:
import os, math
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from einops import repeat


# ------------------------------------------------------------------
# (0) S4 implementation (S4DKernel + S4D)
# ------------------------------------------------------------------
class S4DKernel(nn.Module):
    def __init__(self, d_model, N=64, dt_min=0.001, dt_max=0.1, lr=None):
        super().__init__()
        H = d_model
        log_dt = torch.rand(H) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        self.register_buffer("log_dt", log_dt)

        log_A_real = torch.log(0.5 * torch.ones(H, N//2))
        A_imag     = math.pi * repeat(torch.arange(N//2), 'n -> h n', h=H)
        self.register_buffer("log_A_real", log_A_real)
        self.register_buffer("A_imag", A_imag)

        C = torch.randn(H, N//2, dtype=torch.cfloat)
        self.register_parameter("C", nn.Parameter(torch.view_as_real(C)))

    def forward(self, L):
        dt = torch.exp(self.log_dt)                   # (H)
        C  = torch.view_as_complex(self.C)            # (H,N/2)
        A  = -torch.exp(self.log_A_real) + 1j*self.A_imag  # (H,N/2)

        dtA   = A * dt.unsqueeze(-1)                  # (H,N/2)
        C_mod = C * (torch.exp(dtA) - 1)/A            # (H,N/2)

        t = torch.arange(L, device=dt.device)         # (L)
        exp_term = torch.exp(dtA.unsqueeze(-1) * t)   # (H,N/2,L)
        K = 2 * torch.einsum('hn,hnl->hl', C_mod, exp_term).real  # (H,L)
        return K

    def latent_forward(self, L):
        dt = torch.exp(self.log_dt)
        A  = -torch.exp(self.log_A_real) + 1j*self.A_imag
        dtA = A * dt.unsqueeze(-1)
        t = torch.arange(L, device=dt.device)
        X = torch.exp(dtA.unsqueeze(-1)*t)            # (H,N/2,L)
        return torch.cat([X.real, X.imag], dim=1)      # (H,N,L)

class S4D(nn.Module):
    def __init__(self, d_model, d_state=64, dropout=0.0, transposed=True):
        super().__init__()
        self.transposed = transposed
        self.kernel = S4DKernel(d_model, N=d_state)
        self.act    = nn.GELU()
        self.drop   = nn.Dropout(dropout) if dropout>0 else nn.Identity()

    def forward(self, u):
        # u: (B,H,L)
        if not self.transposed:
            u = u.transpose(-1,-2)              # → (B,L,H)
        L = u.size(-1)
        k = self.kernel(L)                     # (H,L)
        k_f = torch.fft.rfft(k, n=2*L)         # (H,L_freq)
        u_f = torch.fft.rfft(u, n=2*L)         # (B,H,L_freq)
        y   = torch.fft.irfft(u_f * k_f, n=2*L)[..., :L]  # (B,H,L)
        y   = self.drop(self.act(y))
        if not self.transposed:
            y = y.transpose(-1,-2)
        return y, None

# ------------------------------------------------------------------
# (1) YOUR HELPERS: align_eeg_and_fmri_sub02, segment_trials_sub02
# ------------------------------------------------------------------
# from drive.MyDrive.nutsh.hackathons.read_fMRI import align_eeg_and_fmri_sub02, segment_trials_sub02

# ------------------------------------------------------------------
# (2) DATASET — returns (eeg, fmri, song)
# ------------------------------------------------------------------
class EEGFMRIDataset(Dataset):
    def __init__(self, subjects, runs, base_dir,
                 trial_duration_s=40.0, ttl_code=768, transform=None):
        self.transform = transform
        self.data = []
        for sub in subjects:
            sub_dir = os.path.join(base_dir)
            for run in runs:
                eeg, sfreq, fmri_4d, TR, music_song = align_eeg_and_fmri_sub02(run, base=sub_dir)
                events_tsv = os.path.join(sub_dir, 'eeg', f"sub-{sub}_task-{run}_events.tsv")
                trials_eeg, trials_fmri, trials_song = segment_trials_sub02(
                    eeg, sfreq, fmri_4d, TR,
                    music_song=music_song,
                    events_tsv=events_tsv,
                    trial_duration_s=trial_duration_s,
                    ttl_code=ttl_code,
                )
                for i in range(trials_eeg.shape[0]):
                    self.data.append((
                        trials_eeg[i],    # (n_chan, samples)
                        trials_fmri[i],   # (X,Y,Z,vols)
                        trials_song[i],   # (samples,3)
                    ))
        print(f"Total trials across all subjects/runs: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        eeg_np, fmri_np, song_np = self.data[idx]
        eeg_tensor  = torch.from_numpy(eeg_np).float()
        fmri_tensor = torch.from_numpy(fmri_np).float()
        song_tensor = torch.from_numpy(song_np).float()
        if self.transform:
            eeg_tensor, fmri_tensor, song_tensor = self.transform(
                eeg_tensor, fmri_tensor, song_tensor
            )
        return eeg_tensor, fmri_tensor, song_tensor

# ------------------------------------------------------------------
# (3) collate_fn — aligns all to (B,C,T)
# ------------------------------------------------------------------
def collate_fn(batch):
    eeg_list, fmri_list, song_list = zip(*batch)
    B = len(batch)
    eeg_batch  = torch.stack(eeg_list,  dim=0)   # (B,Ce,Le)
    fmri_batch = torch.stack(fmri_list, dim=0)   # (B,X,Y,Z,V)
    song_batch = torch.stack(song_list, dim=0)   # (B,Ls,3)

    _, X,Y,Z, T = fmri_batch.shape
    C_fmri = X*Y*Z

    eeg_res   = F.interpolate(eeg_batch, size=T, mode='linear', align_corners=False)
    fmri_flat = fmri_batch.reshape(B, C_fmri, T)
    song_res  = F.interpolate(song_batch.permute(0,2,1),
                              size=T, mode='linear', align_corners=False
                             ).permute(0,2,1)

    return eeg_res, fmri_flat, song_res      # all (B,C,T)

# ------------------------------------------------------------------
# (4) Two-tower + regression head with S4 front-ends
# ------------------------------------------------------------------
class TwoTowerRegS4(nn.Module):
    def __init__(self, eeg_dim, fmri_dim, song_dim,
                 latent_dim=128, reg_hidden=64,
                 s4_state=64, dropout=0.1):
        super().__init__()
        # --- S4 temporal encoders ---
        self.s4_eeg  = S4D(eeg_dim,  d_state=s4_state, dropout=dropout)
        self.s4_fmri = S4D(fmri_dim, d_state=s4_state, dropout=dropout)

        # --- 2-tower projections ---
        self.eeg_proj  = nn.Linear(eeg_dim,  latent_dim)
        self.fmri_proj = nn.Linear(fmri_dim, latent_dim)

        # --- concat regressor ---
        self.regressor = nn.Sequential(
            nn.Linear(2*latent_dim, reg_hidden),
            nn.ReLU(),
            nn.Linear(reg_hidden, song_dim)
        )

    def forward(self, eeg_seq, fmri_seq):
        # eeg_seq: (B, Ce, T)    fmri_seq: (B, Cf, T)
        eeg_enc, _  = self.s4_eeg(eeg_seq)     # → (B, Ce, T)
        fmri_enc, _ = self.s4_fmri(fmri_seq)   # → (B, Cf, T)

        # collapse into per-timepoint vectors
        B, Ce, T = eeg_enc.shape
        Cf, _     = fmri_enc.shape[1], fmri_enc.shape[2]
        eeg_vecs  = eeg_enc .permute(0,2,1).reshape(-1, Ce)   # (B*T,Ce)
        fmri_vecs = fmri_enc.permute(0,2,1).reshape(-1, Cf)  # (B*T,Cf)

        # two-tower projection + normalize
        # z_e = F.normalize(self.eeg_proj(eeg_vecs),  p=2, dim=-1)
        # z_f = F.normalize(self.fmri_proj(fmri_vecs), p=2, dim=-1)
        z_e = self.eeg_proj(eeg_vecs)
        z_f = self.fmri_proj(fmri_vecs)
        # concat & regress
        z_cat     = torch.cat([z_e, z_f], dim=-1)           # (B*T,2*latent)
        song_pred = self.regressor(z_cat)                   # (B*T, song_dim)
        return song_pred

# ------------------------------------------------------------------
# (5) Main: train/val split, train + evaluate, plot both curves
# ------------------------------------------------------------------
if __name__ == "__main__":
    subjects = [f"{i:02d}" for i in [2]]
    runs     = ["genMusic01","genMusic02","genMusic03"]
    base_dir = BASE_PATH

    full_ds = EEGFMRIDataset(subjects, runs, base_dir)
    n       = len(full_ds)
    n_val   = max(1, int(0.2 * n))
    n_train = n - n_val

    train_ds, val_ds = random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, collate_fn=collate_fn)
    
    # Print train_loader to file
    with open('train_loader.txt', 'w') as f:
        for batch in train_loader:
            print(batch, file=f)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    with torch.no_grad():
        eeg_s, fmri_s, song_s = next(iter(train_loader))
        Ce = eeg_s.shape[1]
        Cf = fmri_s.shape[1]
        _, T, Cs = song_s.shape

    model = TwoTowerRegS4(
        eeg_dim=Ce, fmri_dim=Cf, song_dim=Cs,
        latent_dim=128, reg_hidden=64,
        s4_state=64, dropout=0.1
    ).to(device)

    optim = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    train_hist, val_hist = [], []
    best_val = float('inf')
    best_epoch = -1

    SAVE_PATH = "best_model.pt"

    epochs = 100
    for ep in range(1, epochs+1):
        # — TRAIN —
        model.train()
        rt = 0.0
        for eeg, fmri, song in train_loader:
            eeg, fmri, song = eeg.to(device), fmri.to(device), song.to(device)
            pred = model(eeg, fmri)                          
            loss = F.mse_loss(pred, song.reshape(-1, Cs))    
            optim.zero_grad(); loss.backward(); optim.step()
            rt += loss.item()
        train_loss = rt / len(train_loader)
        train_hist.append(train_loss)

        # — VALIDATE —
        model.eval()
        rv = 0.0
        with torch.no_grad():
            for eeg, fmri, song in val_loader:
                eeg, fmri, song = eeg.to(device), fmri.to(device), song.to(device)
                pred = model(eeg, fmri)
                rv += F.mse_loss(pred, song.reshape(-1, Cs)).item()
        val_loss = rv / len(val_loader)
        val_hist.append(val_loss)

        print(f"Epoch {ep:02d} | Train MSE={train_loss:.2f} | Val MSE={val_loss:.2f}")

        # — CHECK FOR BEST & SAVE —
        if val_loss < best_val:
            best_val = val_loss
            best_epoch = ep
            torch.save(model.state_dict(), SAVE_PATH)
            print(f"  ↳ New best model (epoch {ep:02d}, Val MSE={val_loss:.2f}) saved to {SAVE_PATH}")

    print(f"\nTraining complete. Best epoch: {best_epoch:02d} with Val MSE={best_val:.2f}")

    # — PLOT —
    plt.figure(figsize=(6,4))
    plt.plot(range(1, epochs+1), train_hist, label="Train", marker='o')
    plt.plot(range(1, epochs+1), val_hist,   label="Val",   marker='s')
    plt.axvline(best_epoch, color='k', linestyle='--', label=f"Best Epoch ({best_epoch})")
    plt.xlabel("Epoch"); plt.ylabel("MSE")
    plt.legend(); plt.grid(True); plt.title("Train vs Validation MSE")
    plt.show()



=== Aligning run: genMusic01 ===
Loaded MIDI features from /Users/francescocenciarelli/Desktop/MIND_Hack/MIND_hackathon/midi_player/midi_features.npz
Created mapping for 216 unique MIDI IDs.
Raw EEG: 47 channels × 600000 samples
Extracted music channel index 38: 'music'
Music channel shape: (600000,)
Amount of non-zero samples after scaling/rounding: 11000
Non-zero values (Song IDs) after sparsification: [752 822 753 753 452 753 543 281 751 283 751]
Amount of non-zero samples after sparsification (num peaks): 11
Creating aligned song feature array...
Created music_song array, shape: (600000, 3)
 After picking 32 EEG channels: 32 channels × 600000 samples
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edg

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 50 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 50.00 Hz
- Upper transition bandwidth: 12.50 Hz (-6 dB cutoff frequency: 56.25 Hz)
- Filter length: 6601 samples (6.601 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s


 After filtering: 32 channels × 600000 samples
 First TTL onset (s): 35.683
 TTL sample index: 35683
 After trim EEG: 32 channels × 564317 samples
 After trim music_song: (564317, 3)
 Loaded fMRI data shape: (64, 64, 37, 274)
 masked fMRI data shape: (64, 64, 37, 274)
 fMRI vols: 274    TR(s): 2.0
 Needed EEG samples: 548000
 After CROP EEG: 548000 samples (cropped)
 After CROP music_song: 548000 samples
✔ Final aligned EEG shape: 32 channels × 548000 samples
✔ Final aligned music_song shape: (548000, 3)
✔ Final fMRI    shape: (64, 64, 37, 274)
Found 24 trial starts at:
 [ 37.307  37.354  83.812  83.816 129.869 129.874 174.794 174.798 221.503
 221.508 265.843 265.847 309.23  309.234 352.802 352.807 396.857 396.861
 443.232 443.236 489.29  489.294 533.513 533.517]
Each trial → 40000 EEG samples | 20 fMRI vols
  • skipping EEG at 533.513s → only 14487 samples
  • skipping EEG at 533.517s → only 14483 samples

Checking for zero-value periods within trials_song...
    Trial 0:
      - Zero

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s


 After filtering: 32 channels × 594000 samples
 First TTL onset (s): 23.118
 TTL sample index: 23118
 After trim EEG: 32 channels × 570882 samples
 After trim music_song: (570882, 3)
 Loaded fMRI data shape: (64, 64, 37, 276)
 masked fMRI data shape: (64, 64, 37, 276)
 fMRI vols: 276    TR(s): 2.0
 Needed EEG samples: 552000
 After CROP EEG: 552000 samples (cropped)
 After CROP music_song: 552000 samples
✔ Final aligned EEG shape: 32 channels × 552000 samples
✔ Final aligned music_song shape: (552000, 3)
✔ Final fMRI    shape: (64, 64, 37, 276)
Found 24 trial starts at:
 [ 24.251  24.255  71.277  71.281 116.284 116.288 162.927 162.931 208.868
 208.872 254.875 254.879 298.698 298.702 342.286 342.291 386.708 386.712
 432.283 432.288 478.259 478.264 525.319 525.323]
Each trial → 40000 EEG samples | 20 fMRI vols
  • skipping EEG at 525.319s → only 26681 samples
  • skipping EEG at 525.323s → only 26677 samples

Checking for zero-value periods within trials_song...
    Trial 0:
      - Zero

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s


 After filtering: 32 channels × 593000 samples
 First TTL onset (s): 25.182
 TTL sample index: 25182
 After trim EEG: 32 channels × 567818 samples
 After trim music_song: (567818, 3)
 Loaded fMRI data shape: (64, 64, 37, 274)
 masked fMRI data shape: (64, 64, 37, 274)
 fMRI vols: 274    TR(s): 2.0
 Needed EEG samples: 548000
 After CROP EEG: 548000 samples (cropped)
 After CROP music_song: 548000 samples
✔ Final aligned EEG shape: 32 channels × 548000 samples
✔ Final aligned music_song shape: (548000, 3)
✔ Final fMRI    shape: (64, 64, 37, 274)
Found 24 trial starts at:
 [ 27.359  27.363  73.317  73.322 118.341 118.345 162.329 162.333 206.302
 206.306 252.76  252.764 297.349 297.353 340.287 340.291 386.779 386.783
 433.022 433.026 478.979 478.984 523.051 523.056]
Each trial → 40000 EEG samples | 20 fMRI vols
  • skipping EEG at 523.051s → only 24949 samples
  • skipping EEG at 523.056s → only 24944 samples

Checking for zero-value periods within trials_song...
    Trial 0:
      - Zero

# Code to Run Visualisation of the Validation

In [ ]:
# … after your training loop and plotting …
# ---- PICK ONE TEST TRIAL ----
model = TwoTowerRegS4(
    eeg_dim=Ce, fmri_dim=Cf, song_dim=Cs,
    latent_dim=128, reg_hidden=64,
    s4_state=64, dropout=0.1
).to(device)

# Load the saved weights
state = torch.load("best_model.pt", map_location=device)
model.load_state_dict(state)


model.eval()
with torch.no_grad():
    # grab a single batch from validation
    eeg, fmri, song_true = next(iter(val_loader))    # shapes: (1, Ce, T), (1, Cf, T), (1, T, 3)
    eeg, fmri = eeg.to(device), fmri.to(device)
    # forward pass: returns (B*T, 3)
    pred_flat = model(eeg, fmri)                     
    B, T, Cs = song_true.shape
    # reshape back to time series
    song_pred = pred_flat.view(B, T, Cs).cpu().numpy()
    song_true = song_true.numpy()


# ---- PLOT TRUE vs. PRED ----
feat_names = ["Avg Pitch","Pitch Variance","Harmonicity"]
time_axis  = np.arange(T)   # or multiply by your TR to get seconds

plt.figure(figsize=(8,6))
for i, name in enumerate(feat_names):
    ax = plt.subplot(3,1,i+1)
    ax.plot(time_axis, song_true[0,:,i], label="True",  linewidth=2)
    ax.plot(time_axis, song_pred[0,:,i], label="Pred", linestyle="--")
    ax.set_ylabel(name)
    ax.legend(loc="upper right")
plt.xlabel("fMRI frame index")
plt.tight_layout()
plt.show()
